# 경구약제 118종 분류기 학습

이 노트북은 경구약제 118종 이미지 분류기의 데이터 로드, 학습, checkpoint 재개, validation 평가와 결과 저장을 수행한다.

## 실행 환경

- **Colab:** GPU 런타임을 선택한 뒤 위에서부터 실행한다. 준비 셀에서 Google Drive 연결, 저장소 clone 또는 pull, 데이터 압축 해제를 수행한다.
- **로컬:** 저장소 루트에서 `uv sync --locked`를 실행하고, 데이터 ZIP(기본 v2)을 `data/training_package_classifier_v2/`에 푼다. 사용할 버전은 준비 셀의 `DATA_VERSION`(`"v1"`/`"v2"`)으로 바꾼다.

## 학습 실행 설정

설정 셀의 `run_name`은 학습 실행을 구분하는 이름이다. 학습 결과는 `<output_dir>/<run_name>/`에 저장된다.

- 로컬: `outputs/<run_name>/`
- Colab: `MyDrive/gitkk_runs/classifier/<run_name>/`

처음에는 `run_name = "smoke_test"`와 작은 `max_epochs` 값으로 전체 학습 흐름이 정상 동작하는지 확인한다. 본 학습에서는 `resnet50_baseline`처럼 실험 목적을 알 수 있는 고유한 이름을 사용한다.

같은 `run_name`의 폴더에 기존 checkpoint나 학습 지표가 있으면 결과 덮어쓰기를 방지하기 위해 새 학습이 중단된다. 기존 학습을 이어가려면 같은 `run_name`을 사용하고 `config.resume_from`에 해당 실행의 `last.pt` 경로를 지정한다.

주요 결과물:

- `best.pt`: validation 성능이 가장 좋은 checkpoint
- `last.pt`: 마지막 epoch의 checkpoint
- `metrics.csv`: epoch별 학습 및 validation 지표
- `resolved_config.json`: 실제 학습에 사용된 설정
- validation 예측 및 오류 분석 파일

In [ ]:
# Colab 준비 (로컬에서는 아무것도 하지 않는다)
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f'Running Environment: {"colab" if IN_COLAB else "local"}')

REPO_URL = "https://github.com/auranova1125/gitkk-pill-detection.git"
REPO_BRANCH = "main"
DATA_VERSION = "v2"  # 기준 모델 v1 재현은 "v1"
DATA_PACKAGE = {"v1": "training_package_classifier", "v2": "training_package_classifier_v2"}[DATA_VERSION]
DATA_ZIP = Path(f"/content/drive/MyDrive/gitkk_archive/classifier_data/1. selected_data/{DATA_VERSION}/{DATA_PACKAGE}.zip")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    repo_dir = Path("/content/gitkk-pill-detection")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "-q"], check=True)
    os.chdir(repo_dir)

    # Drive에서 바로 읽으면 느리므로 Colab 로컬에 푼다
    colab_data_dir = Path(f"/content/{DATA_PACKAGE}")
    if not (colab_data_dir / "class_map.json").exists():
        if not DATA_ZIP.exists():
            raise FileNotFoundError(
                f"{DATA_ZIP}가 없습니다. 공유받은 gitkk_archive 폴더라면 Drive에서 '바로가기 추가 → 내 드라이브'를 한 뒤 다시 실행하세요."
            )
        subprocess.run(["unzip", "-q", "-o", str(DATA_ZIP), "-d", str(colab_data_dir)], check=True)

In [ ]:
import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd
from torch import nn

from classifier.evaluate import (
    evaluate_with_predictions,
    save_validation_artifacts,
)
from classifier.inference import load_classifier_inference_model
from classifier.train import train_classifier
from classifier.training_data import prepare_classifier_data
from configs.classifier import ClassifierConfig

In [ ]:
IN_COLAB = "google.colab" in sys.modules

run_name = "smoke_test"
data_factory_path = "data_loading.classifier:build_classifier_inputs"

if IN_COLAB:
    data_dir = Path(f"/content/{DATA_PACKAGE}")
    output_root = Path("/content/drive/MyDrive/gitkk_runs/classifier")  # 세션이 끊겨도 남도록 Drive에 저장
else:
    data_dir = Path(os.environ.get("CLASSIFIER_DATA_DIR", project_root / "data" / DATA_PACKAGE))
    output_root = project_root / "outputs"

config = ClassifierConfig(
    run_name=run_name,
    output_dir=output_root,
    class_map_path=data_dir / "class_map.json",
    data_summary_path=data_dir / "data_summary.json",
    show_progress=True,
    max_epochs=2,
    head_only_epochs=1,
    data_factory_path=data_factory_path,
    expected_num_classes=118,
    pretrained=True,
    device="auto",
)

display_data_dir = os.path.relpath(data_dir, start=project_root)
display_run_dir = os.path.relpath(output_root / run_name, start=project_root)

print(
    f"====================\n"
    f"run_name: {run_name}\n"
    f"====================\n"
)
print(
    f"device: {config.device}\n"
    f"data_dir: {display_data_dir}\n"
    f"run_dir: {display_run_dir}"
)
config_view = config.to_dict()
for key in ["batch_size", "max_epochs", "head_only_epochs", "early_stopping_patience"]:
    print(f"{key}: {config_view[key]}")


## 데이터 연결

`data_factory_path`가 가리키는 함수는 `factory(config=config, device=device)` 호출을 지원하고 `(train_loader, val_loader, class_map, data_summary)`를 반환해야 한다. 실제 데이터 모듈 경로가 확정되면 설정 셀의 문자열만 변경한다.

In [ ]:
training_data = prepare_classifier_data(config)
train_loader = training_data.train_loader
val_loader = training_data.val_loader
class_map = training_data.class_map
data_summary = training_data.data_summary
device = training_data.device

In [ ]:
print(f"입력 검증 완료: device={device}, classes={class_map['num_classes']}")

## 학습 또는 재개

재개할 때는 `config.resume_from`을 해당 실행의 `last.pt`로 설정한 뒤 같은 셀을 실행한다.

In [ ]:
# config.resume_from = output_root / run_name / "last.pt"
training_result = train_classifier(
    config,
    data=training_data,
)
training_result

In [ ]:
metrics = pd.read_csv(training_result.run_dir / "metrics.csv")
plot_metrics = metrics.copy()
plot_metrics["epoch"] += 1

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_metrics.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0, 0])
plot_metrics.plot(
    x="epoch",
    y=["train_top1_accuracy", "val_top1_accuracy"],
    ax=axes[0, 1],
)
plot_metrics.plot(
    x="epoch",
    y=["train_top5_accuracy", "val_top5_accuracy"],
    ax=axes[1, 0],
)
plot_metrics.plot(x="epoch", y="val_macro_f1", ax=axes[1, 1])

for ax in axes.flat:
    ax.set_xlabel("Epoch")
    ax.set_xticks(plot_metrics["epoch"])

fig.tight_layout()

## Best checkpoint 평가와 오류 저장

In [ ]:
best_model, device = load_classifier_inference_model(
    training_result.best_checkpoint_path,
    class_map,
    device=device,
)
artifacts = evaluate_with_predictions(
    best_model,
    val_loader,
    class_map,
    nn.CrossEntropyLoss(),
    device,
    show_progress=config.show_progress,
    progress_description="Classifier final validation",
)
save_validation_artifacts(artifacts, training_result.run_dir)
artifacts.metrics.to_dict()

In [ ]:
class_metrics = pd.DataFrame(
    [metric.to_dict() for metric in artifacts.metrics.per_class]
)
display(class_metrics.sort_values(["support", "recall"], ascending=[True, True]).head(20))

error_samples = pd.DataFrame(
    [sample.to_dict() for sample in artifacts.error_samples]
)
display(error_samples.head(20))

## 유사 사진 제외 점수

validation 중 `near_dup_in_train=True`인 샘플은 train에 거의 같은 사진이 있다 (v1 799개, v2 778개). 전체 val과 이 샘플을 뺀 val(v1 2,064개, v2 2,030개)의 점수를 함께 기록한다.

In [ ]:
import json

from sklearn.metrics import f1_score

predictions = pd.DataFrame(json.loads((training_result.run_dir / "val_predictions.json").read_text()))
near_dup = pd.read_csv(data_dir / "samples.csv", usecols=["sample_id", "near_dup_in_train"])
predictions = predictions.merge(near_dup, on="sample_id")

for name, subset in [("전체 val", predictions), ("유사 사진 제외 val", predictions[~predictions.near_dup_in_train])]:
    macro_f1 = f1_score(subset.target_class_index, subset.predicted_class_index, average="macro")
    print(f"{name}: {len(subset)}개, top1 {subset.correct.mean():.3f}, Macro F1 {macro_f1:.3f}")

## AI Hub val 외부 평가

학습이 끝난 run의 `best.pt`를 AI Hub val(조합) 외부 평가셋으로 평가한다. 실험끼리는 이 평가셋의 **Macro F1(평가셋에 있는 54종 평균)**으로 비교한다. best epoch 선택은 위의 v2 val로 이미 끝났다.

- 드라이브에 저장된 `<output_dir>/<run_name>/best.pt`를 읽으므로, **런타임이 끊겼으면 준비·import·설정 셀(1~3번)만 실행한 뒤 바로 이 셀을 실행**하면 된다. 학습을 다시 돌릴 필요가 없다
- 결과는 `<output_dir>/<run_name>/aihub_val/`에 저장된다 (`evaluation_report.md`, 오분류 목록 등)
- 실험 기록(`reports/experiments/classifier_experiments.md`)에 Top-1, Macro F1, 닮은 사진 제외 Macro F1, 오분류 수와 `eval_hash`를 적는다
- 이 평가셋은 학습에 넣지 않는다. 점수를 보며 설정을 반복해 고치면 평가셋에도 과적합된다


In [ ]:
import subprocess

from classifier.evaluate_external import evaluate_external_package

run_dir = output_root / run_name  # 설정 셀 값으로 run 폴더를 찾는다 (학습 셀을 다시 돌리지 않아도 됨)
best_path = run_dir / "best.pt"
if not best_path.exists():
    raise FileNotFoundError(f"{best_path}가 없습니다. 설정 셀의 run_name을 확인하세요.")

if IN_COLAB:
    EVAL_ZIP = Path("/content/drive/MyDrive/gitkk_archive/classifier_data/1. selected_data/eval_aihub_val/eval_package_aihub_val.zip")
    eval_dir = Path("/content/eval_package_aihub_val")
    if not (eval_dir / "eval_summary.json").exists():
        subprocess.run(["unzip", "-q", "-o", str(EVAL_ZIP), "-d", str(eval_dir)], check=True)
else:
    eval_dir = Path(os.environ.get("CLASSIFIER_EVAL_DIR", project_root / "data" / "classifier" / "export" / "eval_package_aihub_val"))

external = evaluate_external_package(best_path, eval_dir, run_dir / "aihub_val")
external_metrics = external.metrics
print(f"run_name: {run_name}")
print(f"eval_hash: {external.eval_summary['eval_hash'][:8]}…")
print(
    f"AI Hub val: Top-1 {external_metrics.top1_accuracy:.4f}, "
    f"Macro F1 {external_metrics.macro_f1:.4f}, "
    f"오분류 {int((~external.predictions.correct).sum())}/{external_metrics.num_samples}"
)
print(
    f"닮은 사진 제외: Top-1 {external.clean_scores['top1_accuracy']:.4f}, "
    f"Macro F1 {external.clean_scores['macro_f1']:.4f} ({external.clean_scores['num_samples']}개)"
)
display(external.confusion_pairs)
